In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

In [2]:
months = ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6','Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']
features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA', 'Percent_Bleaching']

In [3]:
df = pd.read_csv("../best_data.csv")

In [4]:
df = df[features]

In [5]:
X = df.drop("Percent_Bleaching", axis=1)
Y = df["Percent_Bleaching"]

In [6]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [7]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# OpTuna

In [8]:
paper_features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']

X_train_num = X_train_raw[paper_features].copy()  # 5 numeric cols only

def cv_no_month(model_factory, X_num, y, cv=4):
    from sklearn.model_selection import KFold

    X_num = X_num.reset_index(drop=True)
    y     = y.reset_index(drop=True)

    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in kf.split(X_num):
        X_num_tr, X_num_val = X_num.iloc[train_idx], X_num.iloc[val_idx]
        y_tr  = y.iloc[train_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].reset_index(drop=True)

        # ----- fit preprocessing on train fold only ⭐ -----
        imputer  = SimpleImputer(strategy='median')
        X_tr_imp = imputer.fit_transform(X_num_tr)
        X_vl_imp = imputer.transform(X_num_val)

        scaler  = StandardScaler()
        X_tr_ss = scaler.fit_transform(X_tr_imp)
        X_vl_ss = scaler.transform(X_vl_imp)

        X_tr_final = pd.DataFrame(X_tr_ss, columns=paper_features)
        X_vl_final = pd.DataFrame(X_vl_ss, columns=paper_features)

        # ----- train + evaluate -----
        model = model_factory()
        model.fit(X_tr_final, y_tr)
        scores.append(model.score(X_vl_final, y_val))  # R²

    return np.mean(scores)

In [9]:
def et_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 50, 300),
        max_depth        = trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    return cv_no_month(
        lambda: ExtraTreesRegressor(**params),
        X_train_num,       # 5 numeric cols
        y_train
    )

et_study = optuna.create_study(direction="maximize")
et_study.optimize(et_objective, n_trials=50)
print("========== ET BEST ==========")
print(et_study.best_params)
print(f"Best R²: {et_study.best_value:.4f}")

[I 2026-05-17 14:44:16,304] A new study created in memory with name: no-name-eac88638-77eb-47d5-9c9c-5f5130fa274d
[I 2026-05-17 14:44:17,377] Trial 0 finished with value: 0.47778640547792944 and parameters: {'n_estimators': 254, 'max_depth': 28, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.47778640547792944.
[I 2026-05-17 14:44:17,990] Trial 1 finished with value: 0.1499788750901314 and parameters: {'n_estimators': 201, 'max_depth': 7, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.47778640547792944.
[I 2026-05-17 14:44:18,686] Trial 2 finished with value: 0.4552038790554688 and parameters: {'n_estimators': 169, 'max_depth': 20, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.47778640547792944.
[I 2026-05-17 14:44:19,395] Trial 3 finished with value: 0.09880737572199341 and parameters: {'n_estimators': 224, 'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with va

========== ET BEST ==========
{'n_estimators': 217, 'max_depth': 35, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best R²: 0.6050


In [10]:
def rf_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 50, 300),
        max_depth         = trial.suggest_int("max_depth", 5, 40),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    return cv_no_month(
        lambda: RandomForestRegressor(**params),
        X_train_num,
        y_train
    )

rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=50)
print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")


[I 2026-05-17 14:45:06,049] A new study created in memory with name: no-name-48cfb9ec-55b6-447a-8121-9e22d92537a4
[I 2026-05-17 14:45:08,186] Trial 0 finished with value: 0.5285873138747861 and parameters: {'n_estimators': 256, 'max_depth': 37, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.5285873138747861.
[I 2026-05-17 14:45:08,713] Trial 1 finished with value: 0.5181279139458643 and parameters: {'n_estimators': 51, 'max_depth': 17, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5285873138747861.
[I 2026-05-17 14:45:09,782] Trial 2 finished with value: 0.5225323701172615 and parameters: {'n_estimators': 123, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.5285873138747861.
[I 2026-05-17 14:45:11,131] Trial 3 finished with value: 0.5277685624577801 and parameters: {'n_estimators': 156, 'max_depth': 34, 'm

========== RF BEST ==========
{'n_estimators': 271, 'max_depth': 38, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best R²: 0.6054


In [11]:
def hgb_objective(trial):
    params = dict(
        max_iter          = trial.suggest_int  ("max_iter",          50, 400),
        max_depth         = trial.suggest_int  ("max_depth",          3,  15),
        learning_rate     = trial.suggest_float("learning_rate",  0.005, 0.1),
        min_samples_leaf  = trial.suggest_int  ("min_samples_leaf",   1,  50),
        l2_regularization = trial.suggest_float("l2_regularization", 0.0, 1.0),
        random_state=42
    )
    return cv_no_month(
        lambda: HistGradientBoostingRegressor(**params),
        X_train_num,
        y_train
    )

hgb_study = optuna.create_study(direction="maximize")
hgb_study.optimize(hgb_objective, n_trials=50)
print("========== HGB BEST ==========")
print(hgb_study.best_params)
print(f"Best R²: {hgb_study.best_value:.4f}")

[I 2026-05-17 14:46:39,391] A new study created in memory with name: no-name-f5eebb59-e781-4336-84ea-cc9e3b60a570
[I 2026-05-17 14:46:40,132] Trial 0 finished with value: 0.44256388430374727 and parameters: {'max_iter': 228, 'max_depth': 5, 'learning_rate': 0.08757562859723379, 'min_samples_leaf': 21, 'l2_regularization': 0.15770424559860952}. Best is trial 0 with value: 0.44256388430374727.
[I 2026-05-17 14:46:41,499] Trial 1 finished with value: 0.44741372011640285 and parameters: {'max_iter': 293, 'max_depth': 8, 'learning_rate': 0.040200387339590235, 'min_samples_leaf': 41, 'l2_regularization': 0.8488902950368473}. Best is trial 1 with value: 0.44741372011640285.
[I 2026-05-17 14:46:43,059] Trial 2 finished with value: 0.4681614442351786 and parameters: {'max_iter': 393, 'max_depth': 6, 'learning_rate': 0.06256231980093291, 'min_samples_leaf': 31, 'l2_regularization': 0.44968800247786234}. Best is trial 2 with value: 0.4681614442351786.
[I 2026-05-17 14:46:43,650] Trial 3 finished 

========== HGB BEST ==========
{'max_iter': 301, 'max_depth': 12, 'learning_rate': 0.08994125055209368, 'min_samples_leaf': 7, 'l2_regularization': 0.7761350638156209}
Best R²: 0.5201


# Start

In [ ]:
# ========== ET BEST ==========
# {'n_estimators': 122, 'max_depth': 29, 'min_samples_leaf': 1, 'max_features': 'log2'}

# ========== RF BEST ==========
# {'n_estimators': 204, 'max_depth': 29, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}

# ========== HGB BEST ==========
# {'max_iter': 358, 'max_depth': 13, 'learning_rate': 0.07091369752390678, 'min_samples_leaf': 1, 'l2_regularization': 0.008609248841581194}


In [12]:
hgb_model = HistGradientBoostingRegressor(
    **hgb_study.best_params,
    random_state=42
)

et_model = ExtraTreesRegressor(
    **et_study.best_params,
    n_jobs=-1,
    random_state=42
)

rf_model = RandomForestRegressor(
    **rf_study.best_params,
    n_jobs=-1, 
    random_state=42
)

# K-Fold

In [13]:
from sklearn.model_selection import KFold
from sklearn.base import clone

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

paper_features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']

X_num = X[paper_features].copy()  # 5 numeric only

cv_models = {
    'ExtraTrees':        et_model,
    'RandomForest':      rf_model,
    'HistGradientBoost': hgb_model,
    'Voting (Ensemble)': VotingRegressor([
        ('et',  et_model),
        ('rf',  rf_model),
        ('hgb', hgb_model)
    ])
}

print("=" * 70)
print("K-Fold Cross Validation Summary (k=5)")
print("=" * 70)
print(f"{'Model':<22}{'Mean R²':>14}{'Mean RMSE':>14}{'Mean MAE':>14}")
print("-" * 70)

for name, base_model in cv_models.items():
    r2_scores, rmse_scores, mae_scores = [], [], []

    for train_idx, test_idx in kf.split(X_num):
        X_num_tr, X_num_te = X_num.iloc[train_idx], X_num.iloc[test_idx]
        y_fold_train = Y.iloc[train_idx].reset_index(drop=True)
        y_fold_test  = Y.iloc[test_idx].reset_index(drop=True)

        # ----- fit preprocessing on train fold only ⭐ -----
        imputer  = SimpleImputer(strategy='median')
        X_tr_imp = imputer.fit_transform(X_num_tr)
        X_te_imp = imputer.transform(X_num_te)

        scaler  = StandardScaler()
        X_tr_ss = scaler.fit_transform(X_tr_imp)
        X_te_ss = scaler.transform(X_te_imp)

        X_train_final = pd.DataFrame(X_tr_ss, columns=paper_features)
        X_test_final  = pd.DataFrame(X_te_ss, columns=paper_features)

        # ----- train & evaluate -----
        model = clone(base_model)
        model.fit(X_train_final, y_fold_train)
        preds = model.predict(X_test_final)

        r2_scores.append(r2_score(y_fold_test, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_fold_test, preds)))
        mae_scores.append(mean_absolute_error(y_fold_test, preds))

    print(f"{name:<22}{np.mean(r2_scores):>14.4f}{np.mean(rmse_scores):>14.4f}{np.mean(mae_scores):>14.4f}")

print("=" * 70)

K-Fold Cross Validation Summary (k=5)
Model                        Mean R²     Mean RMSE      Mean MAE
----------------------------------------------------------------------
ExtraTrees                    0.6498       12.7446        5.4918
RandomForest                  0.6454       12.8228        6.3790
HistGradientBoost             0.5375       14.6436        8.4396
Voting (Ensemble)             0.6429       12.8677        6.6443
